In [ ]:
# Step 1: Setup and Data Loading
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Suppressed warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Load the data (adjust path according to your path)
train = pd.read_csv('/kaggle/input/datasets/ebadnaeem/titanic-dataset/Titanic/Dataset/train.csv')
test = pd.read_csv('/kaggle/input/datasets/ebadnaeem/titanic-dataset/Titanic/Dataset/test.csv')

# Quick inspection
print("Training set shape:", train.shape)
print("Test set shape:", test.shape)
print("\nFirst 5 rows of training data:")
print(train.head())
print("\nColumn names and data types:")
print(train.info())

In [ ]:
# Step 2: Data Cleaning and Preparation

# Make copies so we don't alter the originals
train_clean = train.copy()
test_clean = test.copy()

# 1. Extract numeric part from Ticket column
import re

def extract_ticket_num(ticket):
    # Find all digits in the ticket string
    nums = re.findall(r'\d+', str(ticket))
    if nums:
        # Join all digit groups and convert to int
        return int(''.join(nums))
    else:
        # If no digits found (rare), return 0
        return 0

train_clean['Ticket_Num'] = train_clean['Ticket'].apply(extract_ticket_num)
test_clean['Ticket_Num'] = test_clean['Ticket'].apply(extract_ticket_num)

# 2. Fill missing Age with median from training set
age_median = train_clean['Age'].median()
train_clean['Age'] = train_clean['Age'].fillna(age_median)
test_clean['Age'] = test_clean['Age'].fillna(age_median)

# 3. Fill missing Fare (test set has 1 missing) with median from training set
fare_median = train_clean['Fare'].median()
test_clean['Fare'] = test_clean['Fare'].fillna(fare_median)

# 4. Fill missing Embarked with the mode (most frequent port) from training set
embarked_mode = train_clean['Embarked'].mode()[0]
train_clean['Embarked'] = train_clean['Embarked'].fillna(embarked_mode)
test_clean['Embarked'] = test_clean['Embarked'].fillna(embarked_mode)

# 5. Encode Sex: male -> 0, female -> 1
train_clean['Sex'] = train_clean['Sex'].map({'male': 0, 'female': 1})
test_clean['Sex'] = test_clean['Sex'].map({'male': 0, 'female': 1})

# Quick verification
print("Missing values in training set after cleaning:")
print(train_clean.isnull().sum())
print("\nMissing values in test set after cleaning:")
print(test_clean.isnull().sum())

print("\nFirst 5 rows of cleaned training data (showing key columns):")
print(train_clean[['PassengerId', 'Ticket', 'Ticket_Num', 'Age', 'Fare', 'Embarked', 'Sex']].head())

In [ ]:
# Step 3: Build Ticket Neighbor Survival feature (Out-of-Fold)

from sklearn.model_selection import StratifiedKFold
import numpy as np

# Set the ticket number difference threshold
TICKET_DIFF = 50

# --- 1. Compute overall survival rate from training set (for fallback) ---
overall_survival_rate = train_clean['Survived'].mean()

# --- 2. Initialize arrays to store our new feature ---
train_neighbor_survival = np.zeros(len(train_clean))
test_neighbor_survival = np.zeros(len(test_clean))

# --- 3. Out-of-Fold computation for training set ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_clean, train_clean['Survived'])):
    print(f"Processing fold {fold+1}/5...")
    
    # Split the data
    fold_train = train_clean.iloc[train_idx]
    fold_val = train_clean.iloc[val_idx]
    
    # For each passenger in the validation set
    for i, idx in enumerate(val_idx):
        passenger_ticket = train_clean.iloc[idx]['Ticket_Num']
        passenger_neighbors = []
        
        # Find neighbors in the fold training set
        for _, row in fold_train.iterrows():
            if abs(row['Ticket_Num'] - passenger_ticket) <= TICKET_DIFF:
                passenger_neighbors.append(row['Survived'])
        
        # Calculate average survival of neighbors
        if passenger_neighbors:
            train_neighbor_survival[idx] = np.mean(passenger_neighbors)
        else:
            train_neighbor_survival[idx] = overall_survival_rate

# --- 4. Compute feature for test set using full training set ---
print("Computing feature for test set...")
for i, row in test_clean.iterrows():
    passenger_ticket = row['Ticket_Num']
    passenger_neighbors = []
    
    # Find neighbors in the full training set
    for _, train_row in train_clean.iterrows():
        if abs(train_row['Ticket_Num'] - passenger_ticket) <= TICKET_DIFF:
            passenger_neighbors.append(train_row['Survived'])
    
    if passenger_neighbors:
        test_neighbor_survival[i] = np.mean(passenger_neighbors)
    else:
        test_neighbor_survival[i] = overall_survival_rate

# --- 5. Add the new feature to our datasets ---
train_clean['Ticket_Neighbor_Survival'] = train_neighbor_survival
test_clean['Ticket_Neighbor_Survival'] = test_neighbor_survival

# Quick check: look at the distribution
print("\nTicket Neighbor Survival feature added to training set.")
print("Summary statistics:")
print(train_clean['Ticket_Neighbor_Survival'].describe())

print("\nFirst 10 rows of training set with new feature:")
print(train_clean[['PassengerId', 'Ticket_Num', 'Survived', 'Ticket_Neighbor_Survival']].head(10))

In [ ]:
# Step 4: Final Feature Engineering, Model Training, and Submission

# --- 1. Add Family Size feature ---
train_clean['Family_Size'] = train_clean['SibSp'] + train_clean['Parch'] + 1
test_clean['Family_Size'] = test_clean['SibSp'] + test_clean['Parch'] + 1

# --- 2. Encode Embarked using one-hot encoding ---
embarked_dummies_train = pd.get_dummies(train_clean['Embarked'], prefix='Embarked')
embarked_dummies_test = pd.get_dummies(test_clean['Embarked'], prefix='Embarked')

# Ensure both train and test have the same dummy columns (test might miss one if all values are same)
# We use reindex with fill_value=0 to align them
embarked_dummies_test = embarked_dummies_test.reindex(columns=embarked_dummies_train.columns, fill_value=0)

# Concatenate the dummy columns to the datasets
train_clean = pd.concat([train_clean, embarked_dummies_train], axis=1)
test_clean = pd.concat([test_clean, embarked_dummies_test], axis=1)

# --- 3. Select final features for modeling ---
feature_cols = [
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Fare',
    'Family_Size',
    'Ticket_Neighbor_Survival'
] + list(embarked_dummies_train.columns)  # e.g., 'Embarked_C', 'Embarked_Q', 'Embarked_S'

# --- 4. Prepare X (features) and y (target) for training ---
X_train = train_clean[feature_cols]
y_train = train_clean['Survived']
X_test = test_clean[feature_cols]

# Quick check: any missing values in our feature set?
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

# --- 5. Train XGBoost model ---
print("\nTraining XGBoost model...")
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

model.fit(X_train, y_train)

# --- 6. Predict on test set ---
predictions = model.predict(X_test)

# --- 7. Create submission file (for the titanic competition) ---
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' created successfully!")
print("First 10 rows of submission:")
print(submission.head(10))

# --- 8. Quick evaluation on training set using cross-validation to estimate performance ---
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"\nCross-validation accuracy on training set: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# Step 5: Quick Tuning - Smaller ticket diff + Regularized XGBoost

print("=== Tuning for better generalization ===\n")

# --- 1. Recompute Ticket Neighbor Survival with a smaller threshold ---
TICKET_DIFF = 30  # Changed from 50 to 30

# Reset the feature columns to avoid conflicts
train_clean = train_clean.drop(columns=['Ticket_Neighbor_Survival'], errors='ignore')
test_clean = test_clean.drop(columns=['Ticket_Neighbor_Survival'], errors='ignore')

overall_survival_rate = train_clean['Survived'].mean()
train_neighbor_survival = np.zeros(len(train_clean))
test_neighbor_survival = np.zeros(len(test_clean))

# Recompute for training (Out-of-Fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(train_clean, train_clean['Survived'])):
    print(f"Recomputing fold {fold+1}/5 with diff={TICKET_DIFF}...")
    fold_train = train_clean.iloc[train_idx]
    for i, idx in enumerate(val_idx):
        passenger_ticket = train_clean.iloc[idx]['Ticket_Num']
        neighbors = fold_train[abs(fold_train['Ticket_Num'] - passenger_ticket) <= TICKET_DIFF]['Survived']
        train_neighbor_survival[idx] = neighbors.mean() if len(neighbors) > 0 else overall_survival_rate

# Recompute for test set
print(f"Recomputing for test set with diff={TICKET_DIFF}...")
for i, row in test_clean.iterrows():
    passenger_ticket = row['Ticket_Num']
    neighbors = train_clean[abs(train_clean['Ticket_Num'] - passenger_ticket) <= TICKET_DIFF]['Survived']
    test_neighbor_survival[i] = neighbors.mean() if len(neighbors) > 0 else overall_survival_rate

# Add back the new feature
train_clean['Ticket_Neighbor_Survival'] = train_neighbor_survival
test_clean['Ticket_Neighbor_Survival'] = test_neighbor_survival

# --- 2. Preparing data again ---
X_train = train_clean[feature_cols]  # feature_cols already defined in Step 4
y_train = train_clean['Survived']
X_test = test_clean[feature_cols]

# --- 3. Training a more regularized XGBoost model ---
print("\nTraining regularized XGBoost model (max_depth=4, reg_alpha=1, reg_lambda=1)...")
model_reg = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,            # Reduced from 5 to prevent overfitting
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,          # L1 regularization
    reg_lambda=1.0,         # L2 regularization
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

model_reg.fit(X_train, y_train)

# --- 4. Predict and create new submission ---
predictions_reg = model_reg.predict(X_test)

submission_reg = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions_reg
})

submission_reg.to_csv('submission_tuned.csv', index=False)
print("\nNew tuned submission 'submission_tuned.csv' created!")

# Quick CV check on this new model
from sklearn.model_selection import cross_val_score
cv_scores_reg = cross_val_score(model_reg, X_train, y_train, cv=5, scoring='accuracy')
print(f"New Cross-validation accuracy: {cv_scores_reg.mean():.4f} (+/- {cv_scores_reg.std():.4f})")

print("\nFirst 10 rows of new submission:")
print(submission_reg.head(10))

In [ ]:
# Step 6: Broader Features + Ensemble Model

print("=== STEP 6: Adding more features and ensembling ===\n")

# --- 1. Extract Title from Name ---
def get_title(name):
    import re
    title_search = re.search(r' ([A-Za-z]+)\.', name)
    if title_search:
        return title_search.group(1)
    return "Unknown"

train_clean['Title'] = train['Name'].apply(get_title)
test_clean['Title'] = test['Name'].apply(get_title)

# Group rare titles into 'Rare'
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
train_clean['Title'] = train_clean['Title'].replace(rare_titles, 'Rare')
test_clean['Title'] = test_clean['Title'].replace(rare_titles, 'Rare')

# One-hot encode Title
title_dummies_train = pd.get_dummies(train_clean['Title'], prefix='Title')
title_dummies_test = pd.get_dummies(test_clean['Title'], prefix='Title')
title_dummies_test = title_dummies_test.reindex(columns=title_dummies_train.columns, fill_value=0)

# --- 2. Additional features ---
train_clean['Fare_Per_Person'] = train_clean['Fare'] / train_clean['Family_Size']
test_clean['Fare_Per_Person'] = test_clean['Fare'] / test_clean['Family_Size']

train_clean['Is_Child'] = (train_clean['Age'] < 18).astype(int)
test_clean['Is_Child'] = (test_clean['Age'] < 18).astype(int)

train_clean['Is_Alone'] = (train_clean['Family_Size'] == 1).astype(int)
test_clean['Is_Alone'] = (test_clean['Family_Size'] == 1).astype(int)

# --- 3. Deck from Cabin (first letter, 'U' for unknown) ---
train_clean['Deck'] = train_clean['Cabin'].astype(str).apply(lambda x: x[0] if x != 'nan' else 'U')
test_clean['Deck'] = test_clean['Cabin'].astype(str).apply(lambda x: x[0] if x != 'nan' else 'U')

deck_dummies_train = pd.get_dummies(train_clean['Deck'], prefix='Deck')
deck_dummies_test = pd.get_dummies(test_clean['Deck'], prefix='Deck')
deck_dummies_test = deck_dummies_test.reindex(columns=deck_dummies_train.columns, fill_value=0)

# --- 4. Concatenate all new features ---
# Recompute feature_cols to include new ones
feature_cols_extended = [
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Fare',
    'Family_Size',
    'Ticket_Neighbor_Survival',
    'Fare_Per_Person',
    'Is_Child',
    'Is_Alone'
] + list(embarked_dummies_train.columns) + list(title_dummies_train.columns) + list(deck_dummies_train.columns)

# Prepare final datasets
X_train_ext = pd.concat([
    train_clean[feature_cols_extended[:11]],  # numeric + ticket neighbor
    embarked_dummies_train,
    title_dummies_train,
    deck_dummies_train
], axis=1)

X_test_ext = pd.concat([
    test_clean[feature_cols_extended[:11]],
    embarked_dummies_test,
    title_dummies_test,
    deck_dummies_test
], axis=1)

# Ensure all columns match perfectly
X_train_ext = X_train_ext.reindex(columns=feature_cols_extended, fill_value=0)
X_test_ext = X_test_ext.reindex(columns=feature_cols_extended, fill_value=0)

print(f"Extended feature set has {len(feature_cols_extended)} features.")
print("Missing values in X_train_ext:", X_train_ext.isnull().sum().sum())
print("Missing values in X_test_ext:", X_test_ext.isnull().sum().sum())

y_train = train_clean['Survived']

# --- 5. Training two models and ensembling their probabilities ---
# Model A: Regularized XGBoost
print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train_ext, y_train)
xgb_probs = xgb_model.predict_proba(X_test_ext)[:, 1]

# Model B: Random Forest (robust, less overfitting)
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_model.fit(X_train_ext, y_train)
rf_probs = rf_model.predict_proba(X_test_ext)[:, 1]

# --- 6. Ensemble: average probabilities, then threshold at 0.5 ---
ensemble_probs = (xgb_probs + rf_probs) / 2
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

# --- 7. Create submission ---
submission_ensemble = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': ensemble_preds
})

submission_ensemble.to_csv('submission_ensemble.csv', index=False)
print("\n✅ Ensemble submission saved as 'submission_ensemble.csv'")

# Quick CV check on XGBoost with extended features
from sklearn.model_selection import cross_val_score
cv_xgb = cross_val_score(xgb_model, X_train_ext, y_train, cv=5, scoring='accuracy')
print(f"XGBoost CV accuracy (extended features): {cv_xgb.mean():.4f} (+/- {cv_xgb.std():.4f})")

cv_rf = cross_val_score(rf_model, X_train_ext, y_train, cv=5, scoring='accuracy')
print(f"Random Forest CV accuracy: {cv_rf.mean():.4f} (+/- {cv_rf.std():.4f})")

print("\nFirst 10 rows of ensemble submission:")
print(submission_ensemble.head(10))

In [ ]:
# Step 7: Find Optimal Probability Threshold

print("=== STEP 7: Finding the best probability threshold ===\n")

# We already have X_train_ext, y_train, and the ensemble model from Step 6.
# But we need to re-predict probabilities on the full training set to find the best threshold.

# Get predictions on the full training set using the ensemble
# We'll use cross-validation predictions to avoid overfitting the threshold.
from sklearn.model_selection import cross_val_predict

# Get cross-validated probabilities for XGBoost
xgb_cv_probs = cross_val_predict(xgb_model, X_train_ext, y_train, cv=5, method='predict_proba')[:, 1]

# Get cross-validated probabilities for Random Forest
rf_cv_probs = cross_val_predict(rf_model, X_train_ext, y_train, cv=5, method='predict_proba')[:, 1]

# Average them for ensemble CV probabilities
ensemble_cv_probs = (xgb_cv_probs + rf_cv_probs) / 2

# Test different thresholds from 0.3 to 0.7
thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
best_threshold = 0.5
best_score = 0

print("Testing thresholds using cross-validation...")
for thresh in thresholds:
    preds = (ensemble_cv_probs >= thresh).astype(int)
    score = accuracy_score(y_train, preds)
    print(f"Threshold {thresh:.2f}: CV Accuracy = {score:.4f}")
    if score > best_score:
        best_score = score
        best_threshold = thresh

print(f"\n✅ Best threshold: {best_threshold:.2f} with CV accuracy: {best_score:.4f}")

# --- Now predict on test set with the best threshold ---
# We already have xgb_probs and rf_probs from Step 6, so we can use them
ensemble_probs = (xgb_probs + rf_probs) / 2
final_preds = (ensemble_probs >= best_threshold).astype(int)

# Create submission with optimized threshold
submission_optimized = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_preds
})

submission_optimized.to_csv('submission_optimized.csv', index=False)
print(f"\n✅ Optimized submission saved as 'submission_optimized.csv'")
print(f"   Using threshold: {best_threshold:.2f}")
print(f"   Predicted survival rate on test set: {final_preds.mean():.3f}")

print("\nFirst 10 rows:")
print(submission_optimized.head(10))

In [ ]:
# Step 8: XGBoost Alone with Different Hyperparameters

print("=== STEP 8: XGBoost Alone with Tuned Parameters ===\n")

# --- Use the extended features from Step 6 ---
X_train_final = X_train_ext.copy()
X_test_final = X_test_ext.copy()
y_train_final = y_train.copy()

# --- Train XGBoost with different parameters ---
print("Training XGBoost with a different parameter set...")
xgb_final = xgb.XGBClassifier(
    n_estimators=300,          # More trees
    max_depth=6,               # Deeper trees (more complex)
    learning_rate=0.05,        # Lower learning rate (slower, more careful)
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.5,             # Less regularization (allows more complexity)
    reg_lambda=0.5,
    min_child_weight=3,        # Minimum child weight (prevents overfitting)
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_final.fit(X_train_final, y_train_final)

# --- Predict on test set ---
xgb_final_probs = xgb_final.predict_proba(X_test_final)[:, 1]
xgb_final_preds = (xgb_final_probs >= 0.5).astype(int)

# --- Create submission ---
submission_final = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': xgb_final_preds
})

submission_final.to_csv('submission_xgb_final.csv', index=False)

# --- Quick evaluation ---
from sklearn.model_selection import cross_val_score
cv_xgb_final = cross_val_score(xgb_final, X_train_final, y_train_final, cv=5, scoring='accuracy')
print(f"XGBoost CV accuracy: {cv_xgb_final.mean():.4f} (+/- {cv_xgb_final.std():.4f})")

# Check predicted survival rate on test set
print(f"Predicted survival rate on test set: {xgb_final_preds.mean():.3f}")

print("\nFirst 10 rows of XGBoost-only submission:")
print(submission_final.head(10))

In [ ]:
# ============================================================================
# VISUALIZATIONS FOR TITANIC PROJECT
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, 
    roc_curve, auc, classification_report,
    accuracy_score
)
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# Setting a beautiful style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# --- 1. USE OOF PREDICTIONS FROM STEP 7 (Unbiased) ---
# These variables should exist from Step 7:
# ensemble_cv_probs (probabilities), y_train (true labels)
# If they don't exist, we rebuild them quickly:
try:
    oof_probs = ensemble_cv_probs
    y_true = y_train
    print("Using OOF predictions from Step 7.")
except NameError:
    print("OOF variables not found. Rebuilding quickly...")
    from sklearn.model_selection import cross_val_predict
    xgb_probs_oof = cross_val_predict(xgb_model, X_train_ext, y_train, cv=5, method='predict_proba')[:, 1]
    rf_probs_oof = cross_val_predict(rf_model, X_train_ext, y_train, cv=5, method='predict_proba')[:, 1]
    oof_probs = (xgb_probs_oof + rf_probs_oof) / 2
    y_true = y_train

# Threshold at 0.5 for hard predictions
oof_preds = (oof_probs >= 0.5).astype(int)

# --- 2. Confusion Matrix ---
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, oof_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Deceased', 'Survived'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix (Out-of-Fold Predictions)', fontsize=16)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: confusion_matrix.png")

# --- 3. ROC Curve with AUC ---
fig, ax = plt.subplots(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_true, oof_probs)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'Ensemble (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=14)
ax.set_ylabel('True Positive Rate', fontsize=14)
ax.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=16)
ax.legend(loc="lower right")
ax.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: roc_curve.png")

# --- 4. Feature Importance (XGBoost) ---
fig, ax = plt.subplots(figsize=(10, 8))
importance = xgb_model.feature_importances_
feature_names = X_train_ext.columns
indices = np.argsort(importance)[::-1][:15]  # Top 15

# Create horizontal bar chart
ax.barh(range(len(indices)), importance[indices], align='center', color='steelblue')
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names[i] for i in indices], fontsize=11)
ax.invert_yaxis()  # Highest at top
ax.set_xlabel('Feature Importance (Gain)', fontsize=14)
ax.set_title('Top 15 Feature Importances (XGBoost)', fontsize=16)
ax.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('/kaggle/working/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: feature_importance.png")

# --- 5. Distribution of your NOVEL Feature (Ticket_Neighbor_Survival) ---
fig, ax = plt.subplots(figsize=(10, 6))
# Plot density for survived vs deceased using the novel feature
survived = train_clean[train_clean['Survived'] == 1]['Ticket_Neighbor_Survival']
deceased = train_clean[train_clean['Survived'] == 0]['Ticket_Neighbor_Survival']

ax.hist(survived, bins=20, alpha=0.6, label='Survived', color='green', density=True)
ax.hist(deceased, bins=20, alpha=0.6, label='Deceased', color='red', density=True)
ax.set_xlabel('Ticket Neighbor Survival Rate', fontsize=14)
ax.set_ylabel('Density', fontsize=14)
ax.set_title('Distribution of Novel Feature: "Ticket Neighbor Survival"', fontsize=16)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/kaggle/working/novel_feature_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: novel_feature_distribution.png")

# --- 6. t-SNE Visualization (Dimensionality Reduction) ---
print("Computing t-SNE (might take 1-2 minutes)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
X_tsne = tsne.fit_transform(X_train_ext.iloc[:500, :])  # Limit to 500 for speed
y_tsne = y_train.iloc[:500]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_tsne, cmap='coolwarm', alpha=0.7, edgecolors='k')
ax.set_title('t-SNE Visualization of Feature Space (n=500)', fontsize=16)
ax.set_xlabel('t-SNE Component 1', fontsize=14)
ax.set_ylabel('t-SNE Component 2', fontsize=14)
legend1 = ax.legend(*scatter.legend_elements(), title="Survived")
ax.add_artist(legend1)
plt.tight_layout()
plt.savefig('/kaggle/working/tsne_visualization.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: tsne_visualization.png")

# --- 7. Print Classification Report TABLE (to copy into markdown) ---
print("\n" + "="*60)
print("CLASSIFICATION REPORT (Out-of-Fold Predictions)")
print("="*60)
report = classification_report(y_true, oof_preds, target_names=['Deceased (0)', 'Survived (1)'], output_dict=True)
print(classification_report(y_true, oof_preds, target_names=['Deceased (0)', 'Survived (1)']))

# Convert to DataFrame for a nice markdown table
report_df = pd.DataFrame(report).transpose()
print(report_df.round(4).to_markdown())

# --- 8. Save the report table as CSV (just in case) ---
report_df.round(4).to_csv('/kaggle/working/classification_report.csv')
print("\n✅ Saved: classification_report.csv")

# 🚢 Titanic Survival Prediction: Ticket Neighbor Survival Approach

[![Python](https://img.shields.io/badge/Python-3.8%2B-blue.svg)](https://www.python.org/)
[![XGBoost](https://img.shields.io/badge/XGBoost-1.5.0-orange.svg)](https://xgboost.readthedocs.io/)
[![Scikit-Learn](https://img.shields.io/badge/Scikit--Learn-1.0.2-green.svg)](https://scikit-learn.org/)
[![Kaggle](https://img.shields.io/badge/Kaggle-Score%200.77990-20BEFF.svg)](https://www.kaggle.com/)

## 📖 Overview
This project presents a novel feature engineering approach to the classic [Kaggle Titanic competition](https://www.kaggle.com/competitions/titanic). Instead of relying solely on standard features (age, sex, class), I hypothesized that passengers with nearby ticket numbers were seated in similar areas of the ship, meaning their collective survival rate acts as a proxy for cabin-level safety conditions. 

To test this, I engineered a "Ticket Neighbor Survival" feature using Out-of-Fold (OOF) cross-validation to prevent data leakage. Combined with an ensemble of XGBoost and Random Forest models, this approach achieved:

- **Kaggle Public Leaderboard Score:** `0.77990` (Top ~25%)
- **Cross-Validation Accuracy:** `0.8530` (±0.0128)
- **Improvement over Gender Baseline:** `+1.43%`

## 💡 The Core Hypothesis
> *"If two passengers have ticket numbers within ±50 of each other, they likely boarded together or were assigned adjacent cabins. Therefore, the average survival rate of these 'ticket neighbors' reflects how dangerous or safe that specific area of the ship was during the sinking."*

## 🧠 Feature Engineering Highlights
### Novel Feature: `Ticket_Neighbor_Survival`
- Extracted the numeric portion from `Ticket` strings using regex (e.g., `"A/5 21171"` → `521171`).
- Used 5-Fold Stratified K-Fold to compute this feature on the training set:
  - For each passenger, find neighbors in the *fold training set* with `|Ticket_Num - Neighbor_Ticket_Num| <= 30`.
  - Assign the mean `Survived` rate of those neighbors.
  - Fallback to overall survival rate (0.384) if no neighbors exist.
- Applied the same logic to the test set using the entire training set.

### Additional Features
| Feature | Description |
| :--- | :--- |
| `Title` | Extracted from `Name` (Mr, Mrs, Miss, Master, Rare) |
| `Family_Size` | `SibSp + Parch + 1` |
| `Is_Child` | `Age < 18` |
| `Is_Alone` | `Family_Size == 1` |
| `Fare_Per_Person` | `Fare / Family_Size` |
| `Deck` | First letter of `Cabin` ('U' for unknown) |

## 📊 Model & Performance
### Ensemble Strategy
- **Model 1:** XGBoost (`max_depth=4`, `reg_alpha=1.0`, `reg_lambda=1.0`, `n_estimators=200`)
- **Model 2:** Random Forest (`max_depth=6`, `min_samples_split=5`, `n_estimators=200`)
- **Final Prediction:** Average of predicted probabilities, threshold at 0.5.

### Results
#### Confusion Matrix (Out-of-Fold)
![Confusion Matrix](images/confusion_matrix.png)

#### ROC Curve
![ROC Curve](images/roc_curve.png)

#### Feature Importance (XGBoost)
![Feature Importance](images/feature_importance.png)

> **Note:** The novel `Ticket_Neighbor_Survival` feature consistently ranks among the Top most important features, validating the hypothesis.

#### Novel Feature Distribution
![Novel Feature Distribution](images/novel_feature_distribution.png)

#### t-SNE Visualization
![t-SNE Visualization](images/tsne_visualization.png)

### Classification Report
| Class | Precision | Recall | F1-Score | Support |
| :--- | :--- | :--- | :--- | :--- |
| Deceased (0) | 0.87 | 0.88 | 0.88 | 549 |
| Survived (1) | 0.81 | 0.80 | 0.81 | 342 |
| **Accuracy** | **0.853** | | | 891 |
| **Macro Avg** | 0.84 | 0.84 | 0.84 | 891 |
| **Weighted Avg** | 0.85 | 0.85 | 0.85 | 891 |

## 🛠️ Technologies Used
- **Language:** Python 3.8+
- **Data Manipulation:** Pandas, NumPy
- **Visualization:** Matplotlib, Seaborn
- **Machine Learning:** Scikit-Learn, XGBoost
- **Environment:** Kaggle Notebooks

## 📁 Project Structure
titanic-ticket-neighbor/
├── notebook/
│ └── titanic_ticket_neighbor.ipynb # Full Kaggle notebook
├── images/
│ ├── confusion_matrix.png
│ ├── roc_curve.png
│ ├── feature_importance.png
│ ├── novel_feature_distribution.png
│ └── tsne_visualization.png
├── data/
│ ├── train.csv
│ └── test.csv
├── submissions/
│ ├── submission_ensemble.csv
│ └── submission_optimized.csv
├── README.md
└── requirements.txt

## 🚀 How to Run
1. Clone this repository:
   ```bash
   git clone https://github.com/[Your-GitHub-Username]/titanic-ticket-neighbor.git
   cd titanic-ticket-neighbor
2. Install dependencies:
   pip install -r requirements.txt
3. Run the Jupyter Notebook or Python script.

## 🔑 Key Takeaways
- Creative feature engineering (even on classic datasets) can yield measurable improvements.
- Out-of-Fold (OOF) cross-validation is critical to prevent data leakage when creating complex features.
- Ensemble models (XGBoost + Random Forest) provide more stable and generalizable predictions than single models.

## 📝 Author
Ebad Naeem
[Github](https://github.com/Adnyeus) | [LinkedIn](https://www.linkedin.com/in/ebad-naeem-7984522b8)

## 🙏 Acknowledgments
- Kaggle for providing the dataset and platform.
- The open-source community for maintaining essential libraries like Scikit-Learn and XGBoost.